In [1]:
!pip install pymupdf
!pip install torch
!pip install sentence-transformers
!pip install langchain langchain-text-splitters
!pip install ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 5.4 MB/s  0:00:04m 5.4 MB/s eta 0:00:01

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached nvidia_nvshmem_cu13-3.4.5-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (2.1 kB)
  Using cached nvidia_cuda_nvrtc-13.0.88-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_runtime-13.0.96-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cufft-12.0.0.61-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cufile-1.15.1.6-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_cupti-13.0.85-py3-none-manylinux_2_25_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_curand-10.4.0.35-py3-none-manylinux_2_27_

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
#Me permettra de faire le chunking: Decouper le texte en des segments sans pour atutant
#perdre le sens avec du chevauchement du texte . Le texte se repete parfois
#pour pouvoir garder le sens

In [12]:
import fitz
from sentence_transformers import SentenceTransformer, CrossEncoder, util

document = fitz.open("Rapport_Metahuman.pdf")
texte_complet = ""

for page in document:
    texte_complet += page.get_text() + "\n"

#On instancie le splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

#La fonction split_text du separateur me permettra de decouper le texte du coup
text_decoupe = text_splitter.split_text(texte_complet)
print(text_decoupe)

#Je fais l'embedding de ma liste de chunks à l'aide d'un bi-encodeur
#Je choisis le multilangue car il a le meilledur compromis
modele_embedding = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

embeddings = modele_embedding.encode(text_decoupe, convert_to_tensor=True)
print(embeddings)

['Developing believable eye-gaze interaction with\nhigh-fidelity virtual humans in virtual reality\nL.Thomas, R.Alexandre, B.Amadou, J.Mael, B.Tom\nEncadrante : Katja Zibrek\n2025-2026\nRésumé\nCe projet de recherche porte sur le développement d’interactions naturelles\nà travers le regard (eye-gaze interaction) avec un agent virtuel en VR. Menée\ntout au long de l’année académique sous la direction de Katja Zibrek, cette\nétude vise à améliorer le réalisme du comportement des avatars.\nLa phase initiale, s’étendant sur le premier semestre, était dédiée à l’établis-\nsement des bases d’une interaction naturelle avec l’humain virtuel. Nous avons\nréussi à avoir des micro-interactions telles que des clignements et mouvements\naléatoires des yeux, tout en instaurant une communication verbale fluide.\nLe second semestre s’est concentré sur l’implémentation des émotions, le\ntempérament et la conversation naturelle. L’enjeu majeur fut de doter le Me-', 'Le second semestre s’est concentré su

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tensor([[-0.0894,  0.0178,  0.0648,  ...,  0.1586, -0.2806, -0.0847],
        [-0.1056,  0.0618, -0.0374,  ...,  0.3105, -0.2186, -0.0031],
        [ 0.0287, -0.0347, -0.2075,  ...,  0.1939, -0.0899,  0.0504],
        ...,
        [-0.0665,  0.0185, -0.1509,  ..., -0.0184,  0.0601, -0.0064],
        [ 0.0074, -0.0254, -0.0565,  ...,  0.2228, -0.0334,  0.0713],
        [-0.1825,  0.0764,  0.0795,  ...,  0.3438, -0.3319, -0.0077]])


In [ ]:
from sentence_transformers import CrossEncoder
import numpy as np
import torch
import ollama
# Je rajoute un cross-encodeur comme reranker me permettant
#d'ameliorer la precision étant donné
# que je n'ai pas de bons résultats.
modele_reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

print("\n--- Base de données prête ! ---")

while True:
    question = input("\nPose ta question (ou tape 'q' pour quitter) : ")

    if question.lower() == 'q':
        break

    vecteur_question = modele_embedding.encode(question)
    scores_bi_encodeur = util.cos_sim(vecteur_question, embeddings)[0]

    top_5 = torch.topk(scores_bi_encodeur, k=5)
    indices_candidats = top_5.indices.tolist()

    paires = [[question, text_decoupe[i]] for i in indices_candidats]
    scores_reranking = modele_reranker.predict(paires)

    meilleur_index_local = np.argmax(scores_reranking)
    meilleur_indice_global = indices_candidats[meilleur_index_local]
    contexte_selectionne = text_decoupe[meilleur_indice_global]

    # Grace à llama qui est un agent local à qui je peux faire des prompts je vais
    #pouvoir réagir
    prompt = f"""Tu es un assistant étudiant. Réponds à la question posée
     en utilisant uniquement les informations contenues dans le contexte
     ci-dessous. Sois précis et concis.
     Si l'information ne s'y trouve pas, réponds que le document ne permet
     pas de répondre.

    Contexte :
    {contexte_selectionne}

    Question :
    {question}

    Réponse :"""

    reponse = ollama.chat(
        model="llama3.2", messages=[{"role": "user", "content": prompt}]
    )

    texte_reponse = reponse["message"]["content"]

    print("\n--- Réponse synthétisée ---")
    print(texte_reponse)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


--- Base de données prête ! ---



Pose ta question (ou tape 'q' pour quitter) :  POUR LE FUTUR,QUE PREVOIR POUR LE PROJET?



--- Réponse synthétisée ---
Pour le futur, il est prévu de continuer à améliorer la conversation naturelle du MetaHuman, notamment en ce qui concerne la reconnaissance des émotions et des mouvements du corps. Les prochaines étapes pourraient inclure l'ajout de détails visuels tels que la résolution des yeux ou la simulation de la bouche pour un effet plus réaliste. De plus, l'expérience de réalité virtuelle pourrait être mise en œuvre pour offrir un environnement de conversation plus immersive et interactif.



Pose ta question (ou tape 'q' pour quitter) :  COMMENT FONCTIONNE LE PLUGIN CONVAI?



--- Réponse synthétisée ---
Le plugin Convai semble être un outil d'IA conversationnelle capable d'intégrer une IA conversationnelle dans un projet. Selon le contexte fourni, voici une réponse concise :

Le plugin Convai permet d'intégrer une IA conversationnelle avec un compromis entre puissance et accessibilité financière, permettant une intégration limitée de 100 requêtes par mois. Cependant, le document ne fournit pas de détails sur la fonctionnalité spécifique du plugin, telles que la façon dont il reconnait les émotions, traite les requêtes ou fonctionne à l'arrière-plan.
